In [ ]:
import sys
sys.path.append('..')

import torch
from source.evaluation.model_loading import load_model_and_tokenizer
from source.evaluation.evaluation import evaluate_model
from source.evaluation.config import EVAL_PARAMS
from source.data_preprocessing import load_and_preprocess_multiwoz

torch.cuda.empty_cache()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Define model paths (adjust to your actual paths)
pretrained_paths = {
    "Baseline DeepSeekMoE": "./checkpoints/baseline/final",
    "DYNMoE baseline": "./checkpoints/dynmoe_baseline/final",
    "Prototype (DeepSeekMoE + DYNMoE routing)": "./checkpoints/dynmoe_routing/final",
}

finetuned_paths = {
    "Baseline DeepSeekMoE (fine‑tuned)": "./checkpoints/baseline-ft/final",
    "DYNMoE baseline (fine‑tuned)": "./checkpoints/dynmoe_baseline-ft/final",
    "Prototype (fine‑tuned)": "./checkpoints/dynmoe_routing-ft/final",
}

In [ ]:
# Generate test sequences (or load if already present)
train_sequences, val_sequences, test_sequences = load_and_preprocess_multiwoz(
    zip_path="MultiWOZ-coref/MultiWOZ2_3.zip",
    sample_size=300,
    random_seed=42
)
print(f"Test sequences available: {len(test_sequences)}")

In [ ]:
# Helper to run one evaluation
def run_evaluation(label, model_path):
    print("=" * 60)
    print(f"EVALUATING: {label}")
    print(f"Model path: {model_path}")
    print("=" * 60)

    model, tokenizer = load_model_and_tokenizer(model_path)
    model = model.to(device)
    model.eval()

    results = evaluate_model(
        model=model,
        tokenizer=tokenizer,
        test_file="test_sequences.txt",
        device=device,
        **EVAL_PARAMS
    )

    print("Evaluation results:")
    for key, value in results.items():
        print(f"  {key}: {value}")
    print()
    return results

# Run all evaluations
all_results = {}
for label, path in pretrained_paths.items():
    all_results[label] = run_evaluation(label, path)

for label, path in finetuned_paths.items():
    all_results[label] = run_evaluation(label, path)

print("All evaluations completed.")

# Create a comparison DataFrame
import pandas as pd
comparison_df = pd.DataFrame(all_results).T
print("\nComparison across models:")
print(comparison_df)

# Optionally save to CSV
comparison_df.to_csv("model_comparison_results.csv")
print("Comparison saved to model_comparison_results.csv")

In [ ]:
# Run all evaluations
# Pre‑trained models
for label, path in pretrained_paths.items():
    run_evaluation(label, path)

# Fine‑tuned models
for label, path in finetuned_paths.items():
    run_evaluation(label, path)

print("All evaluations completed.")